In [ ]:
import requests
from bs4 import BeautifulSoup
import datetime
import time
import pandas as pd

{'Pos': '1', 'Name': 'Chris MINTERN (#299)', 'Gun Time': '04:01:24', 'Category (Pos)': '30-34(1)', 'Gender (Pos)': 'Male(1)', 'Swim': '00:25:49', 'T1': '00:02:36', 'Cycle': '02:10:21', 'T2': '00:02:47', 'Run': '01:19:48', '': ''}
{'Pos': '2', 'Name': 'Bryan MCCRYSTAL (#4)', 'Gun Time': '04:06:49', 'Category (Pos)': '40-44(1)', 'Gender (Pos)': 'Male(2)', 'Swim': '00:32:51', 'T1': '00:03:26', 'Cycle': '02:05:05', 'T2': '00:02:45', 'Run': '01:22:40', '': ''}
{'Pos': '3', 'Name': 'Loughlin CAMPION (#84)', 'Gun Time': '04:07:50', 'Category (Pos)': '30-34(2)', 'Gender (Pos)': 'Male(3)', 'Swim': '00:30:52', 'T1': '00:02:56', 'Cycle': '02:10:21', 'T2': '00:02:23', 'Run': '01:21:17', '': ''}
{'Pos': '4', 'Name': 'Mark MC GINLEY (#55)', 'Gun Time': '04:22:44', 'Category (Pos)': '25-29(1)', 'Gender (Pos)': 'Male(4)', 'Swim': '00:31:39', 'T1': '00:03:22', 'Cycle': '02:21:16', 'T2': '00:02:45', 'Run': '01:23:40', '': ''}
{'Pos': '5', 'Name': 'Dave HIGGINS (#448)', 'Gun Time': '04:23:25', 'Category 

In [ ]:
def parse_table_from_url(url):
    response = requests.get(url)
    response.raise_for_status()  # make sure request is successful

    soup = BeautifulSoup(response.text, 'html.parser')

    # Locate the table inside the 'table-responsive' container or directly the table
    table = soup.find('table')
    if not table:
        print("No table found on the page.")
        return []

    # Parse table headers
    headers = [th.get_text(strip=True) for th in table.find_all('th')]

    # Parse table rows
    data = []
    for row in table.find_all('tr')[1:]:  # skip header row
        cols = [td.get_text(strip=True) for td in row.find_all('td')]
        if len(cols) == len(headers):
            entry = dict(zip(headers, cols))
            data.append(entry)
    return data

In [ ]:
def extract_positions(df):
    # Extract number from 'Name', e.g. 'Dave Higgins (#56)'
    df['Number'] = df['Name'].str.extract(r'\(#(\d+)\)')
    df['Name'] = df['Name'].str.replace(r'\s*\(#\d+\)', '', regex=True)

    # Extract 'Category' and 'Category_Pos' from 'Category (Pos)', e.g. '25-29(1)'
    cat_extract = df['Category (Pos)'].str.extract(r'(.+?)\((\d+)\)')
    df['Category'] = cat_extract[0].str.strip()
    df['Category_Pos'] = cat_extract[1]
    df.drop(columns=['Category (Pos)'], inplace=True)

    # Extract 'Gender' and 'Gender_Pos' from 'Gender (Pos)', e.g. 'Male(1)'
    gender_extract = df['Gender (Pos)'].str.extract(r'(.+?)\((\d+)\)')
    df['Gender'] = gender_extract[0].str.strip()
    df['Gender_Pos'] = gender_extract[1]
    df.drop(columns=['Gender (Pos)'], inplace=True)

    # Optionally drop any empty column if present
    if '' in df.columns:
        df.drop(columns=[''], inplace=True)

    return df

# Convert time string to timedelta
def str_to_timedelta(t):
    if not t or t.strip() == '':
        return pd.NaT
    parts = t.split(':')
    parts = [int(p) for p in parts]
    if len(parts) == 3:
        return datetime.timedelta(hours=parts[0], minutes=parts[1], seconds=parts[2])
    elif len(parts) == 2:
        return datetime.timedelta(minutes=parts[0], seconds=parts[1])
    return pd.NaT

def add_delta_to_best(df, segments=['Swim', 'T1', 'Cycle', 'T2', 'Run']):
    for seg in segments:
        # Find the best (minimum) time in the column
        best_time = df[seg].min()

        # Compute delta relative to best time for each row
        df[f'{seg}_Delta'] = df[seg] - best_time

    return df

def calculate_percentiles(df, segments=['Swim', 'T1', 'Cycle', 'T2', 'Run'], rank_suffix='_Rank'):
    total = len(df)
    for seg in segments:
        rank_col = f'{seg}{rank_suffix}'
        percentile_col = f'{seg}_Percentile'
        if rank_col in df.columns:
            df[percentile_col] = ((total - df[rank_col] + 1) / total) * 100
    return df

In [ ]:
# Example usage
url = 'https://www.sportsplits.com/races/lost-sheep-triathlon-2025/events/1?page='
raw_results=list()
page=1
while True:
    raw_result = parse_table_from_url(f"{url}{page}")
    if len(raw_result)==0:
        break
    raw_results.append(raw_result)
    page+=1
    if page > 15:
        break
    time.sleep(1)  # be polite and avoid overwhelming the server


In [ ]:
flattened_raw_results = [result for sublist in raw_results for result in sublist]
df_raw_results = pd.DataFrame(flattened_raw_results)
df_raw_results.to_csv('lost_sheep_2025_raw_results.csv', index=False)  # Save raw results to CSV

In [ ]:
df_raw_results = pd.read_csv('lost_sheep_2025_raw_results.csv', dtype=str)
df = df_raw_results
df = extract_positions(df)

# Convert timing columns to timedelta dtype
time_cols = ['Gun Time','Swim', 'T1', 'Cycle', 'T2', 'Run']
for col in time_cols:
    df[col] = df[col].apply(str_to_timedelta)

# Add ranking columns per segment
for col in time_cols:
    rank_col = f'{col}_Rank'
    df[rank_col] = df[col].rank(method='min').astype('Int64')

df = add_delta_to_best(df)
df = calculate_percentiles(df)

AttributeError: 'float' object has no attribute 'strip'

In [87]:
df.to_csv('lost_sheep_2025_results.csv', index=False)

In [88]:
df.to_pickle('lost_sheep_2025_results.pkl')

In [89]:
# read dict from pickle
df = pd.read_pickle('lost_sheep_2025_results.pkl')

In [90]:
name='Maura Barry'
my_result = df[df['Name'] == name]
my_result

,Pos,Name,Club,Gun Time,Swim,T1,Cycle,T2,Run,Number,...,Swim_Delta,T1_Delta,Cycle_Delta,T2_Delta,Run_Delta,Swim_Percentile,T1_Percentile,Cycle_Percentile,T2_Percentile,Run_Percentile
22,23,Maura Barry,,0 days 05:02:47,0 days 00:38:08,0 days 00:03:31,0 days 02:41:36,0 days 00:04:17,0 days 01:35:12,NaN,...,0 days 00:12:05,0 days 00:02:47,0 days 00:27:35,0 days 00:02:10,0 days 00:13:44,68.653422,95.80574,96.02649,88.962472,94.481236


In [86]:
my_result = df[df['Name'] == 'James Carron']
my_result

,Pos,Name,Club,Gun Time,Swim,T1,Cycle,T2,Run,Number,...,Swim_Delta,T1_Delta,Cycle_Delta,T2_Delta,Run_Delta,Swim_Percentile,T1_Percentile,Cycle_Percentile,T2_Percentile,Run_Percentile
115,116,James Carron,,0 days 05:54:55,0 days 00:30:27,0 days 00:07:06,0 days 03:18:56,0 days 00:06:31,0 days 01:51:52,NaN,...,0 days 00:04:24,0 days 00:06:22,0 days 01:04:55,0 days 00:04:24,0 days 00:30:24,98.013245,52.980132,64.01766,62.693157,77.262693


In [77]:
my_result = df[df['Name'] == 'Camila Monteiro']
my_result

,Pos,Name,Club,Gun Time,Swim,T1,Cycle,T2,Run,Number,...,Swim_Rank,T1_Rank,Cycle_Rank,T2_Rank,Run_Rank,Swim_Delta,T1_Delta,Cycle_Delta,T2_Delta,Run_Delta
164,165,Camila Monteiro,,0 days 06:16:07,0 days 00:38:49,0 days 00:05:46,0 days 03:22:26,0 days 00:04:23,0 days 02:04:42,NaN,...,166,137,182,54,173,0 days 00:12:46,0 days 00:05:02,0 days 01:08:25,0 days 00:02:16,0 days 00:43:14
